# Week 6 · U-Net and transfer learning

**Segmentation** assigns a label to every pixel, producing a mask that shows
where objects are. Classification gives one label for the whole image. Here,
each pixel is labelled **foreground (shape)** or **background**.

![Segmentation example: pixel masks outline objects in a street scene.](figures/segmentation.png)

Build a U-Net, reuse a pretrained encoder, and compare freezing with fine-tuning.
Assumes Week 5 CNNs and PyTorch training loops.

Complete the **TODOs** and the written questions. Predict before running;
rerun cells in order after edits. Switch driver after activity 2.

Uses NumPy, Matplotlib, and CPU PyTorch. Data and figures are local; no GPU needed.


In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(2)
torch.manual_seed(6)
print("Ready: PyTorch", torch.__version__, "| CPU")


## 1. Build a small U-Net

Segment foreground/background in 32×32 grayscale shape images.

### Encoder–decoder

- **Encoder:** convolutions extract features; pooling reduces spatial size.
- **Bottleneck:** lowest-resolution features with broader context.
- **Decoder:** upsampling and convolutions restore pixelwise predictions.
  A final 1×1 convolution produces one foreground logit per pixel.

**Discuss:** why does segmentation need a decoder, whereas classification can
use global average pooling and a linear head?


### Upsampling

- **Nearest-neighbour interpolation:** copies values; no learned weights. Used here.
- **Transposed convolution:** upsamples with a learnable kernel.

**Predict the matrix output:** 2× enlargement repeats each value in a 2×2 block.
Batch and channel counts stay fixed.


In [ ]:
matrix = torch.tensor([[1., 2.], [3., 4.]])
enlarged_matrix = F.interpolate(matrix[None, None], scale_factor=2, mode="nearest")
print("Before (2×2):\n", matrix)
print("After (4×4):\n", enlarged_matrix[0, 0])

**Apply the same operation to a photo.** Each grayscale pixel becomes a 2×2 block.
Does the enlarged outline gain detail? Edit a matrix value, then try
`scale_factor=4` below. Display smoothing is disabled.

*Photo: [Abhijit Tembhekar, “Red Apple”](https://commons.wikimedia.org/wiki/File:Red_Apple.jpg),
[CC BY 2.0](https://creativecommons.org/licenses/by/2.0/); grayscale, resized to 32×29.*


In [ ]:
from pathlib import Path

image_path = Path("figures/apple-small.png") 

if not image_path.exists():
    image_path = Path("exercises/week6") / image_path 
x = torch.tensor(plt.imread(image_path))[None, None] 
interpolated = F.interpolate(x, scale_factor=2, mode="nearest")

for image, title in [(x, "Small apple photo"), (interpolated, "Nearest-neighbour interpolation")]:
    h, w = image.shape[-2:]
    plt.figure(figsize=(3.5, 3.5))
    plt.imshow(image[0, 0], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    plt.title(f"{title} ({h}×{w})")
    plt.tight_layout()
    plt.show()


### Transposed convolution: a small example

Take a **2×2 input**, a **2×2 kernel**, and **stride 1**, with no padding or bias:

$$
X=\begin{bmatrix}1&2\\3&4\end{bmatrix},
\qquad
K=\begin{bmatrix}1&2\\3&4\end{bmatrix}.
$$

**Each input value multiplies the entire kernel.** Place its patch at that
input's position and add overlapping values:

$$
\underbrace{\begin{bmatrix}
1&2&0\\3&4&0\\0&0&0
\end{bmatrix}}_{\text{input }1:\;1K}
+
\underbrace{\begin{bmatrix}
0&2&4\\0&6&8\\0&0&0
\end{bmatrix}}_{\text{input }2:\;2K}
+
\underbrace{\begin{bmatrix}
0&0&0\\3&6&0\\9&12&0
\end{bmatrix}}_{\text{input }3:\;3K}
+
\underbrace{\begin{bmatrix}
0&0&0\\0&4&8\\0&12&16
\end{bmatrix}}_{\text{input }4:\;4K}
=
\boxed{\begin{bmatrix}
1&4&4\\6&20&16\\9&24&16
\end{bmatrix}}.
$$

All four patches overlap at the center:

$$
Y_{1,1}=1(4)+2(3)+3(2)+4(1)=20.
$$

With zero-based indices and stride $s$:

$$
\boxed{Y_{r,c}=\sum_{i,j}X_{i,j}\,K_{r-si,\;c-sj}}
$$

Kernel indices outside its bounds contribute zero. Here each output dimension
is $(2-1)\times1+2=\mathbf{3}$.


### Transposed convolution on the photo

Stride 2 separates the 2×2 patches, so they do not overlap.
All-one weights match nearest-neighbour interpolation; weights are not trained here.
**Try `fill_(0.5)`:** which output changes, and why?


In [ ]:
upconv = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2, bias=False)
with torch.no_grad():
    upconv.weight.fill_(1.0)  # Try 0.5.
transposed = upconv(x)

plt.figure(figsize=(3.5, 3.5))
plt.imshow(transposed[0, 0].detach(), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
plt.title(f"Transposed convolution ({transposed.shape[-2]}×{transposed.shape[-1]})")
plt.tight_layout()
plt.show()


### Why skip connections?

Pooling loses spatial detail; resizing cannot recover it. Skips concatenate
**encoder features before pooling** with decoder features at matching resolution.
Decoder convolutions combine local detail with deeper context to locate boundaries.

**Discuss:** why join `skip2` with the upsampled features, rather than `bottom`? What might happen
to thin objects or boundaries without skips? Why can't smoother interpolation
replace these features?

_Your observations:_ …


### The full U-Net

![Our U-Net: two encoder stages, a 32-channel bottleneck, nearest-neighbour upsampling, two skip concatenations, and a one-channel output.](figures/image.png)

We will now build the U-Net shown above.

**Before coding:** use the figure to complete the table (ignore the batch dimension).
Identify the decoder stage receiving each skip in the figure above.

| Stage | Channels × height × width |
|---|---|
| Input | 1 × 32 × 32 |
| Encoder `skip1` | 8 × 32 × 32 |
| Pool → encoder `skip2` | 16 × 16 × 16 |
| Pool → `bottom` | 32 × 8 × 8 |
| Upsample `bottom` | 32 × 16 × 16 |
| Concatenate with `skip2` | ? × 16 × 16 |
| Decoder block 2 | 16 × 16 × 16 |
| Upsample → concatenate with `skip1` | ? × 32 × 32 |
| Decoder block 1 | 8 × 32 × 32 |
| 1×1 output convolution | ? × 32 × 32 |

1. U-Net **concatenates channels**; Week 5 residual blocks added tensors.
   Why must spatial sizes match? Must channel counts match?
2. Why can't a classifier's global average pooling followed by a linear layer
   directly produce a detailed segmentation mask?

_Your table entries and answers:_ …


In [ ]:
# Supplied: two convolutions per resolution, with padding to preserve H and W.
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 3, padding=1), nn.ReLU(),
    )

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.level1 = conv_block(1, 8)
        self.level2 = conv_block(8, 16)
        self.bottom = conv_block(16, 32)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        skip1 = self.level1(x)
        skip2 = self.level2(self.pool(skip1))
        bottom = self.bottom(self.pool(skip2))
        return skip1, skip2, bottom


In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.decode2 = conv_block(32 + 16, 16)
        self.decode1 = conv_block(16 + 8, 8)
        self.output = nn.Conv2d(8, 1, kernel_size=1)

    def forward(self, skip1, skip2, bottom):
        x = F.interpolate(bottom, size=skip2.shape[-2:], mode="nearest")
        # TODO 1: concatenate x and skip2 along the channel dimension (dim=1).
        x = None
        if x is None:
            return None
        x = self.decode2(x)
        x = F.interpolate(x, size=skip1.shape[-2:], mode="nearest")
        # TODO 2: concatenate x and skip1 along the channel dimension.
        x = None
        if x is None:
            return None
        return self.output(self.decode1(x))  # Raw logits.


In [ ]:
class TinyUNet(nn.Module):
    def __init__(self, encoder, decoder=None):
        super().__init__()
        self.encoder = encoder
        self.decoder = Decoder() if decoder is None else decoder

    def forward(self, x):
        skip1, skip2, bottom = self.encoder(x)
        return self.decoder(skip1, skip2, bottom)

probe_model = TinyUNet(Encoder())
probe = probe_model(torch.zeros(2, 1, 32, 32))
unet_ready = probe is not None
if unet_ready:
    assert probe.shape == (2, 1, 32, 32)
    # Check concatenation also works for a different even spatial size.
    assert probe_model(torch.zeros(1, 1, 40, 40)).shape == (1, 1, 40, 40)
    print("U-Net output shapes checked.")
else:
    print("Complete both concatenations, then rerun this cell.")


Use `BCEWithLogitsLoss` with **raw logits** and floating-point 0/1 masks of the
same shape. It includes sigmoid; apply sigmoid separately only to display probabilities.

![Intersection over Union: overlap area divided by union area.](figures/iou.png)

**Pen and paper:** 1 = foreground, 0 = background.

$$
\text{Prediction}=\begin{bmatrix}1&1&0\\1&0&0\\0&0&0\end{bmatrix},\qquad
\text{Ground truth}=\begin{bmatrix}1&0&0\\1&1&0\\0&0&0\end{bmatrix}.
$$

Count TP, FP, and FN, then compute foreground $\mathrm{IoU}=TP/(TP+FP+FN)$.

**Separately:** if 90% of pixels are background, what accuracy and foreground
IoU would an all-background prediction obtain?


In [ ]:
segmentation_loss = nn.BCEWithLogitsLoss()
# Zero logits give probability 0.5, hence mean loss log(2).
test_logits = torch.zeros(1, 1, 2, 2, requires_grad=True)
test_mask = torch.tensor([[[[0., 1.], [1., 0.]]]])
test_loss = segmentation_loss(test_logits, test_mask)
torch.testing.assert_close(test_loss, torch.tensor(np.log(2), dtype=torch.float32))
test_loss.backward()
assert test_logits.grad is not None and torch.isfinite(test_logits.grad).all()
assert test_logits.grad[0, 0, 0, 0] > 0 and test_logits.grad[0, 0, 0, 1] < 0
print("Loss value and gradient direction checked.")

_Your metric calculations and explanation:_ …


### Train the segmentation model

Train a fresh U-Net on **100 synthetic shape masks** and evaluate on
**64 independent validation images**. Later, transfer learning gets only
**25 target masks** drawn from this larger training set.

Shapes vary more in size and position, with lower contrast, uneven background
lighting, and stronger target noise. Masks still cover the whole object.

**Predict:** will validation BCE decrease and IoU increase after 10 epochs?


In [ ]:
# Supplied helpers. Each example contains exactly one nonempty foreground object.
def make_shapes(n, seed, noise):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:32, :32]
    labels = np.arange(n) % 2
    rng.shuffle(labels)
    masks = np.zeros((n, 1, 32, 32), dtype=np.float32)
    for i, label in enumerate(labels):
        cy, cx = rng.integers(8, 24, size=2)
        radius = int(rng.integers(4, 9))
        if label == 0:
            mask = (yy - cy)**2 + (xx - cx)**2 <= radius**2
        else:
            mask = (np.abs(yy - cy) <= radius) & (np.abs(xx - cx) <= radius)
        masks[i, 0] = mask
    # Vary object contrast and background lighting independently of the label.
    brightness = rng.uniform(.35, .9, size=(n, 1, 1, 1))
    background = rng.uniform(.05, .2, size=(n, 1, 1, 1))
    slope = rng.uniform(-.15, .15, size=(n, 1, 1, 1))
    background = background + slope * (xx[None, None] / 31 - .5)
    images = background + brightness * masks + rng.normal(0, noise, masks.shape)
    return (torch.tensor(images, dtype=torch.float32),
            torch.tensor(labels, dtype=torch.long), torch.tensor(masks))

target_train = make_shapes(25, seed=20, noise=.22)
target_valid = make_shapes(64, seed=21, noise=.22)
# The small target set is included in the larger scratch-training set.
extra_train = make_shapes(75, seed=30, noise=.22)
scratch_train = tuple(torch.cat((small, extra))
                      for small, extra in zip(target_train, extra_train))
print(f"Scratch demo: {len(scratch_train[0])} masks | "
      f"Transfer comparison: {len(target_train[0])} masks | "
      f"Validation: {len(target_valid[0])} images")

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for i in range(4):
    axes[0, i].imshow(target_train[0][i, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title("Disk" if target_train[1][i] == 0 else "Square")
    axes[1, i].imshow(target_train[2][i, 0], cmap="gray", vmin=0, vmax=1)
    for ax in axes[:, i]:
        ax.axis("off")
axes[0, 0].set_ylabel("Image")
fig.suptitle("Target images (top) and binary masks (bottom)")
fig.tight_layout()
plt.show()

def train_epochs(model, images, targets, optimizer, loss_fn, epochs, batch_size, seed):
    generator = torch.Generator().manual_seed(seed)
    model.train()
    for _ in range(epochs):
        order = torch.randperm(len(images), generator=generator)
        for ids in order.split(batch_size):
            optimizer.zero_grad(set_to_none=True)
            logits = model(images[ids])
            loss = loss_fn(logits, targets[ids])
            loss.backward()
            optimizer.step()

@torch.no_grad()
def segmentation_metrics(model, data):
    model.eval()
    logits = model(data[0])
    prediction, truth = logits > 0, data[2].bool()  # Probability threshold 0.5.
    dims = (1, 2, 3)
    intersection = (prediction & truth).sum(dims).float()
    union = (prediction | truth).sum(dims).float()
    # Every ground-truth mask is nonempty, so union is positive.
    return nn.functional.binary_cross_entropy_with_logits(logits, data[2]).item(), (intersection / union).mean().item()


In [ ]:
quick_model = None
loss_ready = segmentation_loss is not None
if unet_ready and loss_ready:
    torch.manual_seed(90)
    quick_encoder = Encoder()
    torch.manual_seed(91)
    quick_model = TinyUNet(quick_encoder)
    quick_before = segmentation_metrics(quick_model, target_valid)
    quick_optimizer = torch.optim.Adam(quick_model.parameters(), lr=.003)
    train_epochs(quick_model, scratch_train[0], scratch_train[2], quick_optimizer,
                 segmentation_loss, epochs=10, batch_size=8, seed=92)
    quick_after = segmentation_metrics(quick_model, target_valid)
    print(f"Scratch training: {len(scratch_train[0])} masks, 10 epochs")
    print(f"Before: validation BCE {quick_before[0]:.3f}, IoU {quick_before[1]:.3f}")
    print(f"After:  validation BCE {quick_after[0]:.3f}, IoU {quick_after[1]:.3f}")

    with torch.no_grad():
        quick_masks = (quick_model(target_valid[0][:2]) > 0).float()
    fig, axes = plt.subplots(2, 3, figsize=(7, 4))
    for row in range(2):
        for col, (images, title) in enumerate([
            (target_valid[0], "Image"), (target_valid[2], "Ground truth"),
            (quick_masks, "Prediction")
        ]):
            axes[row, col].imshow(images[row, 0], cmap="gray", vmin=0, vmax=1)
            axes[row, col].axis("off")
            if row == 0:
                axes[row, col].set_title(title)
    plt.tight_layout()
    plt.show()
else:
    print("Complete the U-Net and loss exercises, then rerun in order.")


**Inspect:** did the metrics improve? Compare one predicted boundary with its
ground truth. Why might a model trained from scratch need more labelled masks?

_Your observations:_ …


## 2. What transfers?

Reuse pretrained features for a target task. Freeze them or fine-tune them.

![Transfer learning from a source classifier to a target classifier.](figures/TL.png)

**In this exercise:**

| | Source | Target |
|---|---|---|
| Domain | Variable-contrast shapes, less noise | Same shapes and lighting, stronger noise |
| Training labels | 256 disk/square image labels | 25 foreground/background masks |
| Task | Recognise the shape | Locate every foreground pixel |

**Goal:** test whether reusing the classification encoder helps segmentation
with **25 masks instead of the earlier 100**. Replace the image classifier
with a U-Net decoder. The final comparison also trains scratch on 25 masks
to compare methods at the same target-label budget.

**Discuss and write:**

1. With many image labels but few masks, why might classification pretraining
   help segmentation? Why might it miss precise boundaries?
2. Would you reuse the convolutional encoder, the two-class linear head, or both?
   Explain the difference between image labels and masks.
3. For a small target dataset, would you try freezing or fine-tuning first?
   Give one reason and one risk. What if source images were very different?

_Your answers:_ …


## 3. Pretrain and reuse an encoder

The train/validation sets use independent seeds. Source training uses image
labels, not masks; source masks only render the images. Reuse the training loop
above and identify its five training steps.


In [ ]:
source_train = make_shapes(256, seed=10, noise=.08)
source_valid = make_shapes(80, seed=11, noise=.08)

@torch.no_grad()
def classification_accuracy(model, data):
    model.eval()
    return (model(data[0]).argmax(1) == data[1]).float().mean().item()



In [ ]:
# Source labels teach recognition; target masks teach localisation.
fig, axes = plt.subplots(2, 3, figsize=(8, 4))
for row, label in enumerate([0, 1]):
    source_id = (source_train[1] == label).nonzero()[0].item()
    target_id = (target_train[1] == label).nonzero()[0].item()
    axes[row, 0].imshow(source_train[0][source_id, 0], cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_title("Source label: " + ["disk", "square"][label])
    axes[row, 1].imshow(target_train[0][target_id, 0], cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("Target: noisier image")
    axes[row, 2].imshow(target_train[2][target_id, 0], cmap="gray", vmin=0, vmax=1)
    axes[row, 2].set_title("Target label: pixel mask")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
class ShapeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.head = nn.Linear(32, 2)

    def forward(self, x):
        _, _, bottom = self.encoder(x)
        return self.head(bottom.mean(dim=(2, 3)))

torch.manual_seed(6)
source_model = ShapeClassifier()
source_optimizer = torch.optim.Adam(source_model.parameters(), lr=.003)
train_epochs(source_model, source_train[0], source_train[1], source_optimizer,
             nn.CrossEntropyLoss(), epochs=50, batch_size=32, seed=60)
print(f"Source validation accuracy: {classification_accuracy(source_model, source_valid):.1%}")
# An independent snapshot: target training must not change the source checkpoint.
pretrained_state = copy.deepcopy(source_model.encoder.state_dict())


Load the source encoder weights and set `requires_grad` using `freeze`.
Discard the classifier head.

**Predict:** does `eval()` freeze weights or disable gradients? Does
`requires_grad=False` change weight values?

Freezing and train/eval mode are separate; this encoder has no BatchNorm or dropout.


In [ ]:
def transferred_encoder(freeze=True):
    encoder = Encoder()
    # TODO 4: replace None with encoder.load_state_dict(pretrained_state).
    loaded = None
    if loaded is None:
        return None
    for parameter in encoder.parameters():
        # TODO 5: Set requires_grad according to freeze
        continue
    return encoder

frozen_encoder = transferred_encoder(freeze=True)
transfer_ready = frozen_encoder is not None
if transfer_ready:
    assert all(not p.requires_grad for p in frozen_encoder.parameters())
    assert all(p.requires_grad for p in transferred_encoder(False).parameters())
    assert all(torch.equal(value, pretrained_state[name])
               for name, value in frozen_encoder.state_dict().items())
    print("Encoder weights copied and freezing flags checked.")
else:
    print("Complete both transfer TODOs, then rerun this cell.")


_Your predictions and explanation:_ …

## 4. Freeze, fine-tune, or start from scratch?

Reuse the earlier **100-mask scratch model** and compare it with three models
trained on **25 masks**, all evaluated on the same 64 validation images.
Decoder learning rate is 0.003 throughout; initial decoder weights are identical.

| Approach | Target masks | Epochs | Target updates | Encoder learning rate |
|---|---|---|---|---|
| Scratch (earlier model) | 100 | 10 | 130 | 0.003 |
| Scratch | 25 | 25 | 100 | 0.003 |
| Frozen | 25 | 25 | 100 | No updates |
| Fine-tuned | 25 | 25 | 100 | 0.0003 |

The three 25-mask runs share the same images and minibatch order.

Source pretraining costs extra compute. Report final-epoch validation BCE and mean
per-image IoU. No checkpoint selection; use an
independent test set for a final assessment.

**Predict:** must transfer beat scratch? Why fine-tune the encoder at a smaller learning rate?

_Your predictions:_ …


In [ ]:
results = {}
if transfer_ready and unet_ready and loss_ready and quick_model is not None:
    large_bce, large_iou = segmentation_metrics(quick_model, target_valid)
    results["Scratch (100)"] = {"model": quick_model, "bce": large_bce,
                              "iou": large_iou,
                              "samples": len(scratch_train[0])}
    print(f"Scratch (100) | validation BCE {large_bce:.3f} | mean IoU {large_iou:.3f}")
    for mode in ["Scratch", "Frozen", "Fine-tuned"]:
        # Independent encoder objects: one experiment cannot alter another.
        torch.manual_seed(90)
        encoder = Encoder() if mode == "Scratch" else transferred_encoder(mode == "Frozen")
        torch.manual_seed(91)  # The same decoder initialisation in all runs.
        model = TinyUNet(encoder)
        decoder_parameters = list(model.decoder.parameters())
        groups = [{"params": decoder_parameters, "lr": .003}]
        encoder_parameters = [p for p in encoder.parameters() if p.requires_grad]
        if encoder_parameters:
            groups.append({"params": encoder_parameters,
                           "lr": .0003 if mode == "Fine-tuned" else .003})
        optimizer = torch.optim.Adam(groups)
        train_epochs(model, target_train[0], target_train[2], optimizer,
                     segmentation_loss, epochs=25, batch_size=8, seed=92)
        bce, iou = segmentation_metrics(model, target_valid)
        results[mode] = {"model": model, "bce": bce, "iou": iou,
                         "samples": len(target_train[0])}
        print(f"{mode + ' (25)':13s} | validation BCE {bce:.3f} | mean IoU {iou:.3f}")
else:
    print("Run the scratch demo and complete the transfer exercises, then rerun in order.")


In [ ]:
if results:
    ncols = 2 + len(results)
    fig, axes = plt.subplots(3, ncols, figsize=(2.3 * ncols, 6))
    for row in range(3):
        axes[row, 0].imshow(target_valid[0][row, 0], cmap="gray", vmin=0, vmax=1)
        axes[row, 1].imshow(target_valid[2][row, 0], cmap="gray", vmin=0, vmax=1)
        for col, run in enumerate(results.values(), start=2):
            with torch.no_grad():
                probability = run["model"](target_valid[0][row:row+1]).sigmoid()[0, 0]
            axes[row, col].imshow(probability, cmap="gray", vmin=0, vmax=1)
        for ax in axes[row]:
            ax.axis("off")
    titles = ["Image", "True mask"] + [
        f"{name.replace(' (100)', '')}\n{run['samples']} masks"
        for name, run in results.items()
    ]
    for ax, title in zip(axes[0], titles):
        ax.set_title(title, fontsize=10)
    fig.suptitle("Validation examples: predicted foreground probabilities")
    fig.tight_layout()
    plt.show()


**Record and explain:**

| Approach | Masks | Validation BCE | Mean foreground IoU |
|---|---|---|---|
| Scratch (100) | 100 | … | … |
| Scratch | 25 | … | … |
| Frozen | 25 | … | … |
| Fine-tuned | 25 | … | … |

1. Compare scratch training with 100 versus 25 masks. How does fine-tuning
   perform relative to each scratch model?
2. Compare one boundary in the plots. Why might an accurate source classifier's
   frozen features still be less useful for segmentation?
3. Does this isolate pretraining's effect? Consider trainable parameters, learning
   rates, source compute, and seeds. Propose one controlled follow-up; avoid
   claiming a universal winner.

_Your observations:_ …

**Individual exit ticket:** in three sentences, explain what weights transferred,
how freezing differs from fine-tuning, and what a U-Net skip connection adds.
For a real RGB pretrained encoder, what would you check about input channels,
scaling, and normalisation?

_Your exit ticket:_ …

**Label budget:** how close does 25-mask fine-tuning get to the earlier
100-mask scratch model? Why does this comparison alone not establish the
number of masks each method requires? Consider optimizer updates as well.


## 5. Extended Exercise: What if we remove the skip connections?

Build a decoder using only the bottleneck: remove both concatenations and
adjust the decoder blocks' input channels. Upsampling still doubles height/width.

Train from scratch on the same **100 masks**, with the same encoder initialisation,
10 epochs, batch size 8, learning rate 0.003, and minibatch order as the earlier
U-Net. Reuse that trained model as the **with-skips** baseline.

**Before running:** how many input channels does each decoder block now need?
Predict what will happen to IoU and boundary detail.


In [ ]:
class DecoderWithoutSkips(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO 6: block mapping bottleneck channels to 16 output channels.
        self.decode2 = None
        # TODO 7: block mapping the previous stage's channels to 8 output channels.
        self.decode1 = None
        self.output = nn.Conv2d(8, 1, kernel_size=1)

    def forward(self, skip1, skip2, bottom):
        if self.decode2 is None or self.decode1 is None:
            return None
        # No encoder feature values enter the decoder except bottom.
        x = F.interpolate(bottom, scale_factor=2, mode="nearest")
        x = self.decode2(x)
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.output(self.decode1(x))

ablation_probe = DecoderWithoutSkips()(None, None, torch.zeros(2, 32, 8, 8))
no_skips_ready = ablation_probe is not None
if no_skips_ready:
    assert ablation_probe.shape == (2, 1, 32, 32)
    print("No-skip decoder output shape checked.")
else:
    print("Complete the two decoder blocks, then rerun.")


In [ ]:
skip_comparison = {}
if no_skips_ready and quick_model is not None and loss_ready:
    torch.manual_seed(90)
    ablation_encoder = Encoder()
    torch.manual_seed(91)
    ablation_model = TinyUNet(ablation_encoder, DecoderWithoutSkips())
    ablation_optimizer = torch.optim.Adam(ablation_model.parameters(), lr=.003)
    train_epochs(ablation_model, scratch_train[0], scratch_train[2], ablation_optimizer,
                 segmentation_loss, epochs=10, batch_size=8, seed=92)
    for name, model in [("With skips", quick_model), ("Without skips", ablation_model)]:
        bce, iou = segmentation_metrics(model, target_valid)
        skip_comparison[name] = {"model": model, "bce": bce, "iou": iou}
        print(f"{name:13s} | validation BCE {bce:.3f} | mean IoU {iou:.3f}")

    fig, axes = plt.subplots(2, 4, figsize=(9, 4))
    with torch.no_grad():
        with_skips = quick_model(target_valid[0][:2]).sigmoid()
        without_skips = ablation_model(target_valid[0][:2]).sigmoid()
    for row in range(2):
        for col, (images, title) in enumerate([
            (target_valid[0], "Image"), (target_valid[2], "Ground truth"),
            (with_skips, "With skips"), (without_skips, "Without skips")
        ]):
            axes[row, col].imshow(images[row, 0], cmap="gray", vmin=0, vmax=1)
            axes[row, col].axis("off")
            if row == 0:
                axes[row, col].set_title(title)
    plt.tight_layout()
    plt.show()
else:
    print("Finish the no-skip decoder and run the earlier scratch demo first.")


| Model | Validation BCE | Foreground IoU |
|---|---|---|
| With skips | … | … |
| Without skips | … | … |

1. Which boundaries differ? Relate your observations to the information each
   decoder receives.
2. The no-skip decoder has narrower inputs. Why does this comparison change
   more than the presence of a direct feature path? Suggest one follow-up.

_Your observations:_ …


### References

- [PyTorch transfer learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial): freezing and fine-tuning pretrained features.
- [Ronneberger, Fischer & Brox (2015), U-Net](https://arxiv.org/abs/1505.04597): encoder–decoder architecture and skip connections.
- [PyTorch BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html): loss inputs and conventions.
- [PyTorch interpolation documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html): resizing modes.

Original classroom exercise with synthetic data. Repository licenses: code MIT;
original prose and exercises CC BY-SA 4.0. Credited third-party assets
retain their stated licenses.
